# NB_Admin_Lineage
A **one-time scan** (run it by hand whenever you want a fresh picture): which tables each notebook, pipeline, semantic model and report uses, and **who uses them**: access (item permissions, workspace roles) and activity (report views from the Activity log, SQL endpoint queries).

**Standalone:** every setting is in the first code cell. No config notebook, no Variable Library, no link to the backup.

**Before running:**
1. Attach a lakehouse (any, with schemas on) as the notebook's default lakehouse, or set `OUTPUT_LH_ABFSS`. The results go to its `Tables/dbo/lineage_*`.
2. Run with `DRY_RUN = True` first: it checks the admin rights and counts workspaces and items, writing nothing.

**Rights:** Fabric Administrator (admin item list, Scanner API, Activity log) plus **Contributor** on every workspace whose items should be parsed or whose lakehouse tables go into the catalog. Tenant settings "Enhance admin APIs responses with detailed metadata" and "… with DAX and mashup expressions" must be on.

**Answer table:** `lineage_table_usage` (lakehouse table × user × path: report view, sql query, access, owner). The other `lineage_*` tables hold the detail.

In [ ]:
# ── Settings: everything the scan needs ─────────────────────────────────────────
DRY_RUN         = False     # True: list the matching workspaces and item counts only (checks the admin rights), write nothing
RUN_ACTIVITY    = True      # read the Activity log (report views, last activity_days)
RUN_SQL         = True      # read SQL endpoint query history (last 30 days)
OUTPUT_LH_ABFSS = ''        # blank = the lakehouse attached to this notebook; or abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<lakehouse>/Tables
FOCUS_TABLES    = []        # tables to report on by user, e.g. ['Dim_Customer'], ['LH_FIN_Gold/*'], ['LH_MD_Gold/dbo/Dim_Mat*']
                            #   pattern on table, lakehouse/table, lakehouse/schema/table or workspace/lakehouse/schema/table
                            #   (* and ?, any case); their lakehouses' SQL query history is read too. [] = no focus section

LINEAGE = {
    'workspaces':       ['*'],             # workspace name patterns (* and ?, any case)
    'exclude':          [],                # name patterns to skip
    'item_types':       ['Notebook', 'DataPipeline', 'SemanticModel', 'Report'],   # definitions read and parsed
    'table_lakehouses': ['LH_*'],          # lakehouses whose tables form the catalog ("the tables we built")
    'sql_endpoints':    ['LH_*Gold*'],     # lakehouses whose SQL endpoint query history is read (['LH_*'] = every lakehouse)
    'activities':       ['ViewReport', 'AnalyzeInExcel', 'ExportReport'],   # Activity log events kept (one call per day each)
    'activity_days':    30,                # days read from the Activity log (it keeps 30)
    'usage_days':       30,                # window for report views and SQL queries in lineage_table_usage
    'helper_patterns': [                   # this repo's own read / write helpers: the table name is their string argument
        (r"\bgold\(\s*['\"](\w+)", 'read'),
        (r"\bsilver\(\s*['\"](\w+)", 'read'),
        (r"\bread_table\(\s*[^,()]+,\s*['\"](\w+)", 'read'),
        (r"\bwrite_gold\(\s*\w+\s*,\s*['\"](\w+)", 'write'),
    ],
    'name_aliases':     [(r'_MLV1$', '_MLV')],   # build name → final name (materialized lake views are renamed)
    'mention_min_len':  4,                 # catalog names shorter than this are not matched as plain mentions
    'threads':          4,                 # definitions read in parallel
    'schema':           'dbo',             # output schema ('' if the lakehouse has no schemas)
    'spn':              None,              # or {'tenant_id':…, 'client_id':…, 'key_vault':'https://….vault.azure.net/', 'secret_name':…}
}

In [ ]:
# ── Helpers: output location, Fabric REST ──────────────────────────────────────────────────────────────
import base64, fnmatch, json, time, uuid
from datetime import date, datetime, timedelta, timezone
import requests

API = 'https://api.fabric.microsoft.com/v1'


def output_root():
    """The output lakehouse root: OUTPUT_LH_ABFSS (with or without /Tables), else '' = the attached default lakehouse."""
    p = (OUTPUT_LH_ABFSS or '').strip().rstrip('/')
    for tail in ('/Tables', '/Files'):
        if p.endswith(tail):
            p = p[:-len(tail)]
    if not p and not DRY_RUN:
        ctx = notebookutils.runtime.context
        if not (ctx.get('defaultLakehouseId') or ctx.get('defaultLakehouseName')):
            raise ValueError('No output lakehouse: attach a lakehouse to this notebook (Explorer → Add data items), '
                             'or set OUTPUT_LH_ABFSS in the settings cell.')
    return p


ROOT = output_root()
for _k in ('spark.sql.parquet.datetimeRebaseModeInWrite', 'spark.sql.parquet.int96RebaseModeInWrite'):
    try:
        spark.conf.set(_k, 'CORRECTED')                          # write old dates as they are, never fail on them
    except Exception:
        pass


class Fabric:
    """Fabric REST as the user (or schedule owner) running the notebook: retries 429 / 5xx, follows long-running operations."""
    def __init__(self):
        self.s = requests.Session()
        self.token()

    def token(self):
        self.s.headers['Authorization'] = 'Bearer ' + notebookutils.credentials.getToken('pbi')

    def call(self, method, path, body=None, tries=6):
        url = path if path.startswith('http') else API + path
        for i in range(tries):
            r = self.s.request(method, url, json=body, timeout=120)
            if r.status_code == 401 and i == 0:
                self.token()
                continue
            if r.status_code == 429 or r.status_code >= 500:
                time.sleep(min(int(r.headers.get('Retry-After') or 2 ** i), 60))
                continue
            if r.status_code == 202 and r.headers.get('Location'):
                return self.wait(r)
            if r.status_code >= 400:
                raise RuntimeError(f'{r.status_code} {method} {path}: {r.text[:300]}')
            return r.json() if r.content else {}
        raise RuntimeError(f'{r.status_code} {method} {path} after {tries} tries: {r.text[:300]}')

    def wait(self, r):
        loc, delay = r.headers['Location'], int(r.headers.get('Retry-After') or 2)
        for _ in range(180):
            time.sleep(delay)
            p = self.s.get(loc, timeout=120)
            if p.status_code == 429:
                delay = int(p.headers.get('Retry-After') or 10)
                continue
            if p.status_code >= 400:
                raise RuntimeError(f'{p.status_code} operation: {p.text[:300]}')
            body = p.json() if p.content else {}
            status = body.get('status')
            if status == 'Succeeded':
                res = self.s.get(loc.rstrip('/') + '/result', timeout=120)
                if res.status_code == 404:          # some operations return no result body
                    return {}
                if res.status_code >= 400:
                    raise RuntimeError(f'{res.status_code} operation result: {res.text[:300]}')
                return res.json() if res.content else {}
            if status in ('Failed', 'Undefined'):
                raise RuntimeError(f"operation {status}: {json.dumps(body.get('error'))[:300]}")
            delay = int(p.headers.get('Retry-After') or delay)
        raise TimeoutError('operation still running after 180 polls')

    def pages(self, path):
        out, url = [], path
        while url:
            j = self.call('GET', url)
            out += j.get('value', [])
            url = j.get('continuationUri')
        return out


def plain(err):
    """An error in plain words for the end-of-run report."""
    e = str(err)
    if e.startswith(('401', '403')):
        return 'no access: the account running the scan needs Contributor (or higher) on this workspace'
    if 'NotSupported' in e or 'not supported' in e.lower() or 'OperationNotSupportedForItem' in e:
        return 'this item cannot be exported through the Fabric API (item type or format not supported)'
    if e.startswith('404'):
        return 'the item was deleted or moved while the scan ran'
    return e[:200]


# ── Admin APIs, optional service principal, Delta writer ──────────────────────────────────────
import re
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pyspark.sql import types as T

PBI        = 'https://api.powerbi.com/v1.0/myorg'
SCANNED_AT = datetime.now(timezone.utc)
TODAY      = SCANNED_AT.date()
notes      = []            # plain-words notes for the end-of-run report


def _spn_token():
    s = LINEAGE['spn']
    secret = notebookutils.credentials.getSecret(s['key_vault'], s['secret_name'])
    r = requests.post(f"https://login.microsoftonline.com/{s['tenant_id']}/oauth2/v2.0/token", timeout=60,
                      data={'grant_type': 'client_credentials', 'client_id': s['client_id'], 'client_secret': secret,
                            'scope': 'https://analysis.windows.net/powerbi/api/.default'})
    r.raise_for_status()
    return r.json()['access_token']


if LINEAGE.get('spn'):
    def _token(self):
        self.s.headers['Authorization'] = 'Bearer ' + _spn_token()
    Fabric.token = _token

fab = Fabric()


def pbi(method, path, body=None, tries=6):
    """Power BI REST (scanner, activity log). Like Fabric.call, but a 202 comes back as is: the scanner polls itself."""
    url = path if path.startswith('http') else PBI + path
    for i in range(tries):
        r = fab.s.request(method, url, json=body, timeout=180)
        if r.status_code == 401 and i == 0:
            fab.token()
            continue
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(min(int(r.headers.get('Retry-After') or 5 * 2 ** i), 300))
            continue
        if r.status_code >= 400:
            raise RuntimeError(f'{r.status_code} {method} {path.split("?")[0]}: {r.text[:300]}')
        return r.json() if r.content else {}
    raise RuntimeError(f'{r.status_code} {method} {path.split("?")[0]} after {tries} tries: {r.text[:300]}')


def fabric_pages(path, key):
    """Fabric admin lists name their rows ('workspaces', 'itemEntities') instead of 'value'."""
    out, url = [], path
    while url:
        j = fab.call('GET', url)
        out += j.get(key, [])
        url = j.get('continuationUri')
    return out


def match(name, pats):
    return any(fnmatch.fnmatch((name or '').lower(), p.lower()) for p in pats)


def is_focus(c):
    """A catalog table matches FOCUS_TABLES on its name, lakehouse/table, lakehouse/schema/table or its full key."""
    sch = f"{c['schema']}/" if c['schema'] else ''
    return bool(FOCUS_TABLES) and any(match(n, FOCUS_TABLES) for n in
        (c['table'], f"{c['lakehouse']}/{c['table']}", f"{c['lakehouse']}/{sch}{c['table']}", c['table_key']))


_TYPES = {'s': T.StringType(), 'l': T.LongType(), 'b': T.BooleanType(), 't': T.TimestampType(), 'd': T.DateType()}
_TS    = re.compile(r'(\d{4}-\d\d-\d\d)[T ](\d\d:\d\d:\d\d)(?:\.(\d+))?')


def to_ts(v):
    """API timestamps ('2026-11-22T10:01:02.1234567Z') → UTC datetime. Python's fromisoformat stops at 6 digits."""
    if v is None or isinstance(v, datetime):
        return v
    m = _TS.match(str(v))
    if not m:
        return None
    return datetime.strptime(f'{m.group(1)} {m.group(2)}', '%Y-%m-%d %H:%M:%S').replace(
        microsecond=int((m.group(3) or '0')[:6].ljust(6, '0')), tzinfo=timezone.utc)


def _cast(v, t):
    if v is None:
        return None
    if t == 's':
        return v if isinstance(v, str) else json.dumps(v) if isinstance(v, (dict, list)) else str(v)
    if t == 'l':
        return int(v)
    if t == 'b':
        return bool(v)
    if t == 't':
        ts = to_ts(v)
        return ts if ts and ts.year >= 1900 else None        # the APIs send '0001-01-01' for "never": store it as empty
    if t == 'd':
        d = v if isinstance(v, date) and not isinstance(v, datetime) else date.fromisoformat(str(v)[:10])
        return d if d.year >= 1900 else None
    return v


def out_path(name):
    sch = LINEAGE.get('schema') or ''
    return f"{ROOT + '/' if ROOT else ''}Tables/{sch + '/' if sch else ''}{name}"   # relative = the default lakehouse


def write_rows(name, rows, cols, mode='overwrite'):
    """cols: 'name:type', type s(tring) l(ong) b(oolean) t(imestamp) d(ate). Every table also gets scanned_at."""
    spec = [c.split(':') for c in cols] + [['scanned_at', 't']]
    schema = T.StructType([T.StructField(n, _TYPES[t], True) for n, t in spec])
    data = [tuple(_cast(SCANNED_AT if n == 'scanned_at' else r.get(n), t) for n, t in spec) for r in rows]
    w = spark.createDataFrame(data, schema).write.format('delta').mode(mode)
    w = w.option('overwriteSchema', 'true') if mode == 'overwrite' else w.option('mergeSchema', 'true')
    try:
        w.save(out_path(name))
    except Exception as e:
        raise RuntimeError(
            f'Could not write {name} to {out_path(name)}. Check that: the lakehouse has schemas turned on; the schema '
            f"'{LINEAGE.get('schema')}' exists in it (Lakehouse explorer → New schema); the account running this is "
            f'Contributor on its workspace. Spark error: {plain(e)[:600]}') from e
    return len(data)


def read_rows(name):
    """An earlier run's output table as a list of dicts, or [] the first time."""
    try:
        return [r.asDict() for r in spark.read.format('delta').load(out_path(name)).collect()]
    except Exception:
        return []

In [ ]:
# ── Parsers: item definitions → table references (pure Python, no Spark: tested outside Fabric) ─────────────────────
#   Each parser returns plain dicts. Names are resolved against the table catalog later (resolve_ref), so a parser
#   never guesses which lakehouse a bare table name belongs to.
import base64, json, re

GUID = r'[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}'


def def_parts(definition):
    """{path: text} for every inline part of a getDefinition response."""
    out = {}
    for p in (definition or {}).get('parts') or []:
        try:
            out[p['path']] = base64.b64decode(p['payload']).decode('utf-8', errors='replace')
        except Exception:
            continue
    return out


def ref(raw, direction, match_type, evidence, **kw):
    """One reference found in an item. raw = the name or path as written."""
    r = dict(raw=raw, direction=direction, match_type=match_type, evidence=evidence,
             ws=None, lh=None, schema=None, table=None, dynamic=False)
    r.update(kw)
    return r


# ── Names and paths ─────────────────────────────────────────────────────────────────────────────────────────────
ONELAKE = re.compile(r'abfss://([^@/\s\'"]+)@onelake\.dfs\.fabric\.microsoft\.com/([^/\s\'"]+)/Tables/([^\s\'"?]+)', re.I)
REL_TABLES = re.compile(r'^/?(?:lakehouse/default/)?Tables/([^\s\'"?]+)$', re.I)


def split_name(raw):
    """'LH.dbo.T' / 'dbo.T' / 'T' / 'ws.LH.dbo.T' (backticks or brackets allowed) → dict(ws, lh, schema, table)."""
    parts = [p.strip('`[]"') for p in re.split(r'\.(?![^`\[]*[`\]])', raw.strip())]
    parts = [p for p in parts if p]
    out = dict(ws=None, lh=None, schema=None, table=None)
    if not parts:
        return out
    out['table'] = parts[-1]
    if len(parts) == 2:
        out['schema'] = parts[0]          # could also be a lakehouse without schemas: resolve_ref tries both
    elif len(parts) == 3:
        out['lh'], out['schema'] = parts[0], parts[1]
    elif len(parts) >= 4:
        out['ws'], out['lh'], out['schema'] = parts[-4], parts[-3], parts[-2]
    return out


def split_path(path):
    """OneLake or relative Tables/ path → dict(ws, lh, schema, table), or None if it is not a table path."""
    m = ONELAKE.search(path)
    if m:
        ws, lh, rest = m.group(1), m.group(2), m.group(3)
    else:
        m = REL_TABLES.match(path.strip())
        if not m:
            return None
        ws, lh, rest = None, None, m.group(1)
    lh = re.sub(r'\.Lakehouse$', '', lh or '', flags=re.I) or None
    segs = [s for s in rest.strip('/').split('/') if s]
    if not segs:
        return None
    if len(segs) >= 2 and not segs[1].startswith('_'):
        return dict(ws=ws, lh=lh, schema=segs[0], table=segs[1])
    return dict(ws=ws, lh=lh, schema=None, table=segs[0])


# ── SQL text (Spark SQL, T-SQL) ─────────────────────────────────────────────────────────────────────────────────
NAME = r'((?:`[^`]+`|\[[^\]]+\]|\{[^}]*\}|[\w\-$]+)(?:\.(?:`[^`]+`|\[[^\]]+\]|\{[^}]*\}|[\w\-$]+)){0,3})'
SQL_WRITE = re.compile(
    r'\b(?:INSERT\s+(?:INTO|OVERWRITE)(?:\s+TABLE)?|MERGE\s+INTO|UPDATE|DELETE\s+FROM|TRUNCATE\s+TABLE|ALTER\s+TABLE'
    r'|CREATE\s+(?:OR\s+REPLACE\s+)?(?:MATERIALIZED\s+LAKE\s+VIEW|TABLE|VIEW|TEMP(?:ORARY)?\s+VIEW)(?:\s+IF\s+NOT\s+EXISTS)?'
    r'|DROP\s+(?:TABLE|VIEW|MATERIALIZED\s+LAKE\s+VIEW)(?:\s+IF\s+EXISTS)?|OPTIMIZE|VACUUM|REFRESH\s+TABLE)\s+' + NAME, re.I)
SQL_READ = re.compile(r'\b(?:FROM|JOIN|USING|TABLE)\s+' + NAME, re.I)
SQL_CTE = re.compile(r'(?:\bWITH|,)\s*(\w+)\s+AS\s*\(', re.I)
SQL_TEMP = re.compile(r'\bCREATE\s+(?:OR\s+REPLACE\s+)?(?:GLOBAL\s+)?TEMP(?:ORARY)?\s+VIEW\s+(?:IF\s+NOT\s+EXISTS\s+)?(\w+)', re.I)
SQL_START = re.compile(r'^\s*(?:--[^\n]*\n\s*)*(SELECT|WITH|INSERT|MERGE|UPDATE|DELETE|CREATE|DROP|ALTER|OPTIMIZE|VACUUM'
                       r'|TRUNCATE|REFRESH|DESCRIBE|SHOW|CACHE)\b', re.I)   # a string literal is SQL only if it starts like SQL
SQL_LOOKS = re.compile(r'\b(SELECT|INSERT|MERGE|UPDATE|DELETE|CREATE|DROP|ALTER|OPTIMIZE|VACUUM|TRUNCATE)\b', re.I)
NOT_TABLES = {'select', 'values', 'lateral', 'unnest', 'json', 'parquet', 'csv', 'delta', 'dual', 'where', 'set',
              'as', 'on', 'the', 'a', 'source', 'target', 'tgt', 'src', 'updates', 'if', 'not', 'exists', 'table',
              'view', 'into', 'openrowset', 'string_split', 'generate_series', 'range', 'explode'}


def sql_refs(text, evidence):
    """Tables read and written by a SQL text. Temp views and CTE names are left out."""
    if not text or not SQL_LOOKS.search(text):
        return []
    text = re.sub(r'--[^\n]*', ' ', text)
    text = re.sub(r'/\*.*?\*/', ' ', text, flags=re.S)
    local = {m.group(1).lower() for m in SQL_CTE.finditer(text)} | {m.group(1).lower() for m in SQL_TEMP.finditer(text)}
    out, taken = [], []

    def add(raw, direction, pos):
        if raw.startswith('('):
            return
        low = raw.strip('`[]').lower()
        if low in NOT_TABLES or low in local or low.isdigit():
            return
        dyn = '{' in raw
        path = re.match(r'delta\.`([^`]+)`', raw, re.I)
        if path:
            p = split_path(path.group(1))
            if p:
                out.append(ref(raw, direction, 'sql path', evidence, **p))
            return
        out.append(ref(raw, direction, 'sql', evidence, dynamic=dyn, **({} if dyn else split_name(raw))))
        taken.append(pos)

    for m in SQL_WRITE.finditer(text):
        add(m.group(1), 'write', m.start(1))
    for m in SQL_READ.finditer(text):
        if m.start(1) in taken:                        # DELETE FROM / MERGE INTO already counted as a write
            continue
        # `FROM x` straight after DELETE / MERGE INTO is the write target, skip
        add(m.group(1), 'read', m.start(1))
    # delta.`path` inside FROM is caught above; abfss paths in the SQL text
    for m in ONELAKE.finditer(text):
        p = split_path(m.group(0))
        if p and not any(o.get('table') == p['table'] for o in out):
            out.append(ref(m.group(0), 'read', 'sql path', evidence, **p))
    return out


# ── Notebooks (.ipynb) ──────────────────────────────────────────────────────────────────────────────────────────
STR = r'([rRfFbBuU]{0,2})(\'\'\'[\s\S]*?\'\'\'|"""[\s\S]*?"""|\'(?:\\.|[^\'\\\n])*\'|"(?:\\.|[^"\\\n])*")'
STR_LIT = re.compile(STR)
PY_CALLS = [   # (pattern before the first string argument, direction, label)
    (r'spark\.read(?:\.\w+\([^()]*\))*\.table\(\s*', 'read', 'spark.read.table'),
    (r'spark\.table\(\s*', 'read', 'spark.table'),
    (r'DeltaTable\.forName\(\s*\w+\s*,\s*', 'read', 'DeltaTable.forName'),
    (r'DeltaTable\.forPath\(\s*\w+\s*,\s*', 'read', 'DeltaTable.forPath'),
    (r'\.load\(\s*', 'read', 'load'),
    (r'\.synapsesql\(\s*', 'read', 'synapsesql'),
    (r'\.saveAsTable\(\s*', 'write', 'saveAsTable'),
    (r'\.insertInto\(\s*', 'write', 'insertInto'),
    (r'\.toTable\(\s*', 'write', 'toTable'),
    (r'\.writeTo\(\s*', 'write', 'writeTo'),
    (r'\.save\(\s*', 'write', 'save'),
]
PY_CALLS = [(re.compile(p + STR), d, lbl) for p, d, lbl in PY_CALLS]
RUN_NB = [re.compile(r'^\s*%run\s+(?:-b\s+\S+\s+)?([^\s#]+)', re.M),
          re.compile(r'(?:notebookutils|mssparkutils)\.notebook\.(?:run|runMultiple)\(\s*' + STR)]


def _unquote(prefix, lit):
    body = lit[3:-3] if lit[:3] in ("'''", '"""') else lit[1:-1]
    return body, ('f' in prefix.lower() and '{' in body)


def _name_or_path(value, dynamic, direction, label, evidence):
    if dynamic:
        return ref(value, direction, label, evidence, dynamic=True)
    p = split_path(value) if ('/' in value or value.lower().startswith('abfss')) else None
    if p:
        return ref(value, direction, label + ' path', evidence, **p)
    if '/' in value:                                     # a Files/ path or a URL: not a table
        return None
    return ref(value, direction, label, evidence, **split_name(value))


def cell_language(cell, nb_lang):
    src = ''.join(cell.get('source') or '')
    first = src.lstrip().split('\n', 1)[0].strip().lower()
    if first.startswith('%%sql'):
        return 'sql'
    if first.startswith(('%%spark', '%%scala')):
        return 'scala'
    if first.startswith('%%sparkr'):
        return 'r'
    meta = (cell.get('metadata') or {}).get('microsoft', {}).get('language') or ''
    return {'sparksql': 'sql', 'sql': 'sql', 'python': 'python', 'scala': 'scala', 'r': 'r'}.get(meta.lower(), nb_lang)


def parse_notebook(ipynb_text, helper_patterns=()):
    """→ (refs, links, info). info has the attached lakehouses and the counts the report needs."""
    nb = json.loads(ipynb_text)
    meta = nb.get('metadata') or {}
    lhmeta = ((meta.get('dependencies') or {}).get('lakehouse')) or {}
    info = dict(default_lh_id=lhmeta.get('default_lakehouse'), default_lh=lhmeta.get('default_lakehouse_name'),
                default_ws_id=lhmeta.get('default_lakehouse_workspace_id'),
                known_lh_ids=[k.get('id') for k in lhmeta.get('known_lakehouses') or [] if k.get('id')],
                cells=0, dynamic=0)
    nb_lang = ((meta.get('language_info') or {}).get('name') or 'python').lower()
    nb_lang = 'sql' if nb_lang in ('sql', 'sparksql') else 'python'
    helpers = [(re.compile(p), d) for p, d in helper_patterns]
    refs, links = [], []
    for i, cell in enumerate(nb.get('cells') or []):
        if cell.get('cell_type') != 'code':
            continue
        src = ''.join(cell.get('source') or '')
        if not src.strip():
            continue
        info['cells'] += 1
        ev = f'cell {i}'
        lang = cell_language(cell, nb_lang)
        for rx in RUN_NB:
            for m in rx.finditer(src):
                target = m.group(m.lastindex) if rx is RUN_NB[0] else _unquote(m.group(1), m.group(2))[0]
                links.append(dict(target=target.split('/')[-1], link='runs notebook', evidence=ev))
        if lang == 'sql':
            refs += sql_refs(re.sub(r'^\s*%%sql[^\n]*', '', src), ev)
            continue
        code = '\n'.join(l for l in src.split('\n') if not l.lstrip().startswith('#'))
        for rx, direction, label in PY_CALLS:
            for m in rx.finditer(code):
                value, dyn = _unquote(m.group(1), m.group(2))
                r = _name_or_path(value, dyn, direction, label, ev)
                if r:
                    refs.append(r)
        for rx, direction in helpers:
            for m in rx.finditer(code):
                refs.append(ref(m.group(1), direction, 'helper', ev, **split_name(m.group(1))))
        for m in STR_LIT.finditer(code):                  # SQL inside string literals (spark.sql("..."), f-strings)
            body, dyn = _unquote(m.group(1), m.group(2))
            if SQL_START.match(body):
                refs += sql_refs(body, ev)
    info['dynamic'] = sum(1 for r in refs if r['dynamic'])
    return refs, links, info


def mention_refs(text, names_rx, already, evidence='text'):
    """Catalog table names that appear as whole words, for tables no explicit rule found."""
    if names_rx is None:
        return []
    found = {}
    for m in names_rx.finditer(text):
        found.setdefault(m.group(0).lower(), m.group(0))
    return [ref(v, 'mention', 'name mention', evidence, table=v) for k, v in found.items() if k not in already]


def build_names_rx(table_names, min_len=4, ignore_case=False):
    """One regex for every catalog table name. Case-sensitive by default: code writes a table name the way it is
    spelled, and that keeps common words out. SQL text is matched with ignore_case=True."""
    names = sorted({n for n in table_names if len(n) >= min_len and re.fullmatch(r'[\w\-$]+', n)}, key=len, reverse=True)
    if not names:
        return None
    return re.compile(r'(?<![\w$])(?:' + '|'.join(re.escape(n) for n in names) + r')(?![\w$])',
                      re.I if ignore_case else 0)


# ── Data pipelines (pipeline-content.json) ──────────────────────────────────────────────────────────────────────
NESTED = ('activities', 'ifTrueActivities', 'ifFalseActivities', 'defaultActivities')


def _val(v):
    """Pipeline values can be {'value': '@...', 'type': 'Expression'}: return (text, dynamic)."""
    if isinstance(v, dict) and 'value' in v:
        v = v['value']
    if not isinstance(v, str):
        return (None, False)
    return (v, v.lstrip().startswith('@'))


def walk_activities(acts, path=''):
    for a in acts or []:
        yield a, (path + ' › ' if path else '') + a.get('name', '?')
        tp = a.get('typeProperties') or {}
        for k in NESTED:
            yield from walk_activities(tp.get(k), (path + ' › ' if path else '') + a.get('name', '?'))
        for case in tp.get('cases') or []:
            yield from walk_activities(case.get('activities'), (path + ' › ' if path else '') + a.get('name', '?'))


def _dataset_ref(ds, direction, evidence):
    """datasetSettings of a Copy / Lookup / GetMetadata activity → a reference."""
    ls = (ds.get('linkedService') or {})
    lsp = (ls.get('properties') or {})
    ltp = lsp.get('typeProperties') or {}
    dtp = ds.get('typeProperties') or {}
    table, tdyn = _val(dtp.get('table') or dtp.get('tableName'))
    schema, sdyn = _val(dtp.get('schema'))
    ws, _ = _val(ltp.get('workspaceId'))
    art, _ = _val(ltp.get('artifactId'))
    kind = lsp.get('type') or ds.get('type') or ''
    if not table:
        folder, fdyn = _val((dtp.get('location') or {}).get('folderPath') or dtp.get('folderPath'))
        if folder and folder.lower().startswith('tables'):
            p = split_path(folder) or {}
            return ref(folder, direction, f'copy {kind}'.strip(), evidence, dynamic=fdyn, ws=ws, lh_id=art,
                       lh=ls.get('name'), schema=p.get('schema'), table=p.get('table'))
        return None
    return ref(f'{schema + "." if schema else ""}{table}', direction, f'copy {kind}'.strip(), evidence,
               dynamic=tdyn or sdyn, ws=ws, lh_id=art, lh=ls.get('name'), schema=schema, table=table)


def parse_pipeline(content_text):
    """→ (refs, links). links: notebooks run, pipelines invoked, semantic models refreshed, dataflows refreshed."""
    doc = json.loads(content_text)
    acts = (doc.get('properties') or doc).get('activities') or []
    refs, links = [], []
    for a, ev in walk_activities(acts):
        t, tp = a.get('type', ''), a.get('typeProperties') or {}
        ev = f'{ev} ({t})'
        if t in ('TridentNotebook', 'SynapseNotebook', 'Notebook'):
            nid, d = _val(tp.get('notebookId'))
            links.append(dict(target_id=nid, target_ws=_val(tp.get('workspaceId'))[0], link='runs notebook',
                              evidence=ev, dynamic=d))
        elif t in ('InvokePipeline', 'ExecutePipeline'):
            pid, d = _val(tp.get('pipelineId') or (tp.get('pipeline') or {}).get('referenceName'))
            links.append(dict(target_id=pid, target_ws=_val(tp.get('workspaceId'))[0], link='invokes pipeline',
                              evidence=ev, dynamic=d))
        elif t in ('PBISemanticModelRefresh', 'SemanticModelRefresh'):
            did, d = _val(tp.get('datasetId'))
            links.append(dict(target_id=did, target_ws=_val(tp.get('groupId') or tp.get('workspaceId'))[0],
                              link='refreshes model', evidence=ev, dynamic=d))
        elif t in ('RefreshDataflow',):
            did, d = _val(tp.get('dataflowId'))
            links.append(dict(target_id=did, target_ws=_val(tp.get('workspaceId'))[0], link='refreshes dataflow',
                              evidence=ev, dynamic=d))
        elif t == 'SparkJobDefinition':
            sid, d = _val(tp.get('sparkJobDefinitionId'))
            links.append(dict(target_id=sid, target_ws=_val(tp.get('workspaceId'))[0], link='runs spark job',
                              evidence=ev, dynamic=d))
        for role in ('source', 'sink'):                  # Copy
            ds = (tp.get(role) or {}).get('datasetSettings')
            if ds:
                r = _dataset_ref(ds, 'read' if role == 'source' else 'write', ev)
                if r:
                    refs.append(r)
        if tp.get('datasetSettings'):                    # Lookup, GetMetadata, Delete
            r = _dataset_ref(tp['datasetSettings'], 'write' if t == 'Delete' else 'read', ev)
            if r:
                refs.append(r)
        texts = [(tp.get('source') or {}).get(k) for k in ('sqlReaderQuery', 'query', 'sqlReaderStoredProcedureName')]
        texts += [s.get('text') for s in tp.get('scripts') or [] if isinstance(s, dict)]
        texts += [(tp.get('sink') or {}).get('preCopyScript')]
        for x in texts:
            s, dyn = _val(x)
            if s:
                for r in sql_refs(s, ev):
                    r['dynamic'] = r['dynamic'] or dyn
                    refs.append(r)
    return refs, links


# ── Semantic models (TMDL) ──────────────────────────────────────────────────────────────────────────────────────
M_SQL = re.compile(r'Sql\.Databases?\(\s*"([^"]+)"\s*(?:,\s*"([^"]+)")?', re.I)
M_ONELAKE = re.compile(r'onelake\.dfs\.fabric\.microsoft\.com/(' + GUID + r'|[^/"\s]+)/(' + GUID + r'|[^/"\s]+)', re.I)
M_LAKEHOUSE = re.compile(r'Lakehouse\.Contents\(', re.I)
M_ITEM = re.compile(r'\{\s*\[\s*Schema\s*=\s*"([^"]+)"\s*,\s*Item\s*=\s*"([^"]+)"\s*\]\s*\}', re.I)
M_ID = re.compile(r'\{\s*\[\s*Id\s*=\s*"([^"]+)"\s*,\s*ItemKind\s*=\s*"Table"', re.I)
M_WSID = re.compile(r'\{\s*\[\s*workspaceId\s*=\s*"(' + GUID + r')"\s*\]\s*\}', re.I)
M_LHID = re.compile(r'\{\s*\[\s*lakehouseId\s*=\s*"(' + GUID + r')"\s*\]\s*\}', re.I)


def _tmdl_name(s):
    s = s.strip()
    if s.startswith("'"):
        return s[1:s.index("'", 1)].replace("''", "'") if "'" in s[1:] else s.strip("'")
    return re.split(r'\s*=\s*|\s', s, maxsplit=1)[0]


def m_source(expr):
    """Where an M expression reads from → dict(server, database, ws, lh, schema, table) (any may be None)."""
    out = dict(server=None, database=None, ws=None, lh=None, schema=None, table=None)
    if not expr:
        return out
    m = M_SQL.search(expr)
    if m:
        out['server'], out['database'] = m.group(1), m.group(2)
    m = M_ONELAKE.search(expr)
    if m:
        out['ws'], out['lh'] = m.group(1), m.group(2)
    m = M_WSID.search(expr)
    if m:
        out['ws'] = m.group(1)
    m = M_LHID.search(expr)
    if m:
        out['lh'] = m.group(1)
    m = M_ITEM.search(expr)
    if m:
        out['schema'], out['table'] = m.group(1), m.group(2)
    m = M_ID.search(expr)
    if m and not out['table']:
        out['table'] = m.group(1)
    return out


def parse_tmdl(parts):
    """parts {path: text} of a TMDL definition → (model_tables, expressions).
    model_tables: one dict per table: name, mode, entity, schema, expression_source, m (import M), columns, measures."""
    exprs = {}
    for path, text in parts.items():
        if path.endswith('expressions.tmdl'):
            cur, buf = None, []
            for line in text.split('\n'):
                m = re.match(r'^expression\s+(.+?)\s*=(.*)$', line)
                if m:
                    if cur:
                        exprs[cur] = '\n'.join(buf)
                    cur, buf = _tmdl_name(m.group(1)), [m.group(2)]
                elif cur is not None:
                    buf.append(line)
            if cur:
                exprs[cur] = '\n'.join(buf)
    tables = []
    for path, text in sorted(parts.items()):
        if not (path.startswith('definition/tables/') and path.endswith('.tmdl')):
            continue
        t = dict(name=None, mode=None, entity=None, schema=None, expression_source=None, m='',
                 columns=[], measures=[], partitions=0, calculated=False)
        in_part, in_source = False, False
        for line in text.split('\n'):
            s = line.strip()
            depth = len(line) - len(line.lstrip('\t'))
            if depth == 0 and s.startswith('table '):
                t['name'] = _tmdl_name(s[6:])
            elif depth == 1:
                in_part = s.startswith('partition ')
                in_source = False
                if s.startswith('column '):
                    t['columns'].append(_tmdl_name(s[7:]))
                elif s.startswith('measure '):
                    t['measures'].append(_tmdl_name(s[8:]))
                elif in_part:
                    t['partitions'] += 1
                    if re.search(r'=\s*calculated\b', s):
                        t['calculated'] = True
            elif in_part:
                if s.startswith('mode:'):
                    t['mode'] = s.split(':', 1)[1].strip()
                elif s.startswith('entityName:'):
                    t['entity'] = s.split(':', 1)[1].strip().strip("'\"")
                elif s.startswith('schemaName:'):
                    t['schema'] = s.split(':', 1)[1].strip().strip("'\"")
                elif s.startswith('expressionSource:'):
                    t['expression_source'] = _tmdl_name(s.split(':', 1)[1])
                elif s.startswith('source') and depth == 2:
                    in_source = True
                    if '=' in s:
                        t['m'] += s.split('=', 1)[1] + '\n'
                elif in_source and depth >= 3 and not s.startswith(('annotation ', 'lineageTag')):
                    t['m'] += s + '\n'
        if t['name']:
            tables.append(t)
    return tables, exprs


def model_sources(tables, exprs):
    """model table → lakehouse source. Direct Lake: entity + schema + the named expression. Import: the M query."""
    out = []
    for t in tables:
        src = m_source(exprs.get(t['expression_source'])) if t['expression_source'] else m_source(t['m'])
        if t['entity']:
            src['table'], src['schema'] = t['entity'], t['schema'] or src['schema']
        out.append(dict(model_table=t['name'], mode=t['mode'] or ('calculated' if t['calculated'] else ''),
                        columns=len(t['columns']), measures=len(t['measures']), **src))
    return out


# ── Reports (PBIR folder or legacy report.json) ─────────────────────────────────────────────────────────────────
def _maybe_json(v):
    if isinstance(v, str) and v[:1] in '{[':
        try:
            return json.loads(v)
        except Exception:
            return v
    return v


def _fields(node, aliases, out):
    """Every (Entity, Property, kind) under a node. Legacy queries name the table through a From alias."""
    node = _maybe_json(node)
    if isinstance(node, dict):
        for f in node.get('From') or []:
            if isinstance(f, dict) and f.get('Name') and f.get('Entity'):
                aliases[f['Name']] = f['Entity']
        for kind in ('Column', 'Measure', 'Aggregation', 'HierarchyLevel'):
            v = node.get(kind)
            if isinstance(v, dict):
                prop = v.get('Property') or v.get('Level')
                e = v.get('Expression') or {}
                if kind == 'Aggregation':
                    inner = e.get('Column') or e.get('Measure') or {}
                    prop, e = inner.get('Property'), inner.get('Expression') or {}
                if kind == 'HierarchyLevel':
                    h = e.get('Hierarchy') or {}
                    e = h.get('Expression') or {}
                    e = (e.get('PropertyVariationSource') or {}).get('Expression') or e
                sr = e.get('SourceRef') or {}
                ent = sr.get('Entity') or aliases.get(sr.get('Source'))
                if ent and prop:
                    out.add((ent, prop, 'measure' if kind == 'Measure' else 'column'))
        for v in node.values():
            _fields(v, aliases, out)
    elif isinstance(node, list):
        for v in node:
            _fields(v, aliases, out)


def parse_report(parts):
    """→ (fields, model_ref). fields: dicts page, visual, visual_type, table, field, kind. model_ref from definition.pbir."""
    fields, model_ref = [], {}
    pbir = parts.get('definition.pbir')
    if pbir:
        try:
            dr = (json.loads(pbir).get('datasetReference') or {})
            cs = ((dr.get('byConnection') or {}).get('connectionString') or '')
            m = re.search(r'semanticmodelid=(' + GUID + ')', cs, re.I)
            model_ref = dict(model_id=m.group(1) if m else None,
                             model_path=((dr.get('byPath') or {}).get('path') or None))
        except Exception:
            pass

    def add(found, page, visual, vtype):
        for ent, prop, kind in sorted(found):
            fields.append(dict(page=page, visual=visual, visual_type=vtype, table=ent, field=prop, kind=kind))

    if 'report.json' in parts and not any(p.startswith('definition/pages/') for p in parts):
        rep = json.loads(parts['report.json'])
        found = set(); _fields(rep.get('filters'), {}, found); add(found, '(report filters)', '', '')
        for sec in rep.get('sections') or []:
            page = sec.get('displayName') or sec.get('name') or ''
            found = set(); _fields(sec.get('filters'), {}, found); add(found, page, '(page filters)', '')
            for vc in sec.get('visualContainers') or []:
                cfg = _maybe_json(vc.get('config')) or {}
                sv = cfg.get('singleVisual') or {}
                found = set(); aliases = {}
                _fields(cfg, aliases, found); _fields(vc.get('filters'), aliases, found)
                add(found, page, cfg.get('name', ''), sv.get('visualType', ''))
        return fields, model_ref

    pages = {}
    for path, text in parts.items():
        m = re.match(r'definition/pages/([^/]+)/page\.json$', path)
        if m:
            try:
                pj = json.loads(text)
            except Exception:
                continue
            pages[m.group(1)] = pj.get('displayName') or m.group(1)
            found = set(); _fields(pj.get('filterConfig'), {}, found); add(found, pages[m.group(1)], '(page filters)', '')
    for path, text in parts.items():
        m = re.match(r'definition/pages/([^/]+)/visuals/([^/]+)/visual\.json$', path)
        if m:
            try:
                vj = json.loads(text)
            except Exception:
                continue
            found = set(); _fields(vj, {}, found)
            add(found, pages.get(m.group(1), m.group(1)), vj.get('name') or m.group(2),
                (vj.get('visual') or {}).get('visualType', ''))
    if 'definition/report.json' in parts:
        try:
            found = set(); _fields(json.loads(parts['definition/report.json']).get('filterConfig'), {}, found)
            add(found, '(report filters)', '', '')
        except Exception:
            pass
    return fields, model_ref

In [ ]:
# ── Workspaces, items and users (Fabric admin item list + Power BI Scanner API) ────────────────────────────────────
#   Admin item list: every item with consistent type names and its creator.
#   Scanner: model tables and expressions, report → model, and every user or group with access (workspace and item).
try:
    all_ws = fabric_pages(f'{API}/admin/workspaces?type=Workspace&state=Active', 'workspaces')
except Exception as e:
    raise RuntimeError('The Fabric admin API refused the call: ' + str(e)[:150] + '\nThe account running the scan must be '
                       'a Fabric Administrator, or set LINEAGE["spn"] to a service principal allowed to use read-only '
                       'admin APIs (tenant setting).')
workspaces = sorted((w for w in all_ws if match(w['name'], LINEAGE['workspaces']) and not match(w['name'], LINEAGE['exclude'])),
                    key=lambda w: w['name'])
ws_name = {w['id']: w['name'] for w in workspaces}

items = {}
for it in fabric_pages(f'{API}/admin/items', 'itemEntities'):
    if it.get('workspaceId') not in ws_name:
        continue
    cp = it.get('creatorPrincipal') or {}
    items[it['id']] = dict(workspace_id=it['workspaceId'], workspace=ws_name[it['workspaceId']], item_id=it['id'],
                           item=it.get('name'), item_type=it.get('type'), state=it.get('state'),
                           created_by=(cp.get('userDetails') or {}).get('userPrincipalName') or cp.get('displayName'),
                           modified_by=None, modified_at=it.get('lastUpdatedDate'), model_id=None, report_type=None)

# ── Scanner: 100 workspaces per scan ──────────────────────────────────────────────────────────────────────────────
SCAN_Q    = 'lineage=True&datasourceDetails=True&datasetSchema=True&datasetExpressions=True&getArtifactUsers=True'
TYPE_KEYS = {'reports': 'Report', 'datasets': 'SemanticModel', 'dashboards': 'Dashboard', 'dataflows': 'Dataflow',
             'datamarts': 'Datamart', 'warehouses': 'Warehouse'}
scan_models = {}       # model id → scanner tables / expressions (fallback when the definition can't be read)
access      = []       # one row per principal per workspace or item


def _principal(u, scope, wid, iid, iname, itype, right):
    return dict(workspace_id=wid, workspace=ws_name.get(wid), item_id=iid, item=iname, item_type=itype, scope=scope,
                principal=u.get('emailAddress') or u.get('identifier') or u.get('displayName'),
                principal_name=u.get('displayName'), principal_type=u.get('principalType'),
                user_type=u.get('userType'), access_right=right)


scan_failed = []
if not DRY_RUN:
    ids = [w['id'] for w in workspaces]
    for k in range(0, len(ids), 100):
        batch = ids[k:k + 100]
        try:
            job = pbi('POST', f'/admin/workspaces/getInfo?{SCAN_Q}', {'workspaces': batch})
            status = ''
            for _ in range(360):                                    # up to 30 minutes per batch
                status = pbi('GET', f"/admin/workspaces/scanStatus/{job['id']}").get('status')
                if status in ('Succeeded', 'Failed'):
                    break
                time.sleep(5)
            if status != 'Succeeded':
                raise RuntimeError(f'scan {status or "timed out"}')
            result = pbi('GET', f"/admin/workspaces/scanResult/{job['id']}")
        except Exception as e:
            scan_failed.append(f'workspaces {k + 1}-{k + len(batch)}: {plain(e)}')
            continue
        for w in result.get('workspaces') or []:
            wid = w['id']
            for u in w.get('users') or []:
                access.append(_principal(u, 'workspace', wid, wid, w.get('name'), 'Workspace', u.get('groupUserAccessRight')))
            for key, val in w.items():
                if key == 'users' or not isinstance(val, list):
                    continue
                itype = TYPE_KEYS.get(key, key)
                for it in val:
                    if not isinstance(it, dict) or 'id' not in it:
                        continue
                    row = items.setdefault(it['id'], dict(workspace_id=wid, workspace=ws_name.get(wid), item_id=it['id'],
                                                          item=it.get('name'), item_type=itype, state=it.get('state'),
                                                          created_by=None, modified_by=None, modified_at=None,
                                                          model_id=None, report_type=None))
                    row['created_by'] = it.get('createdBy') or it.get('configuredBy') or row['created_by']
                    row['modified_by'] = it.get('modifiedBy') or row['modified_by']
                    row['modified_at'] = it.get('modifiedDateTime') or it.get('lastUpdatedDate') or row['modified_at']
                    if itype == 'Report':
                        row['model_id'], row['report_type'] = it.get('datasetId'), it.get('reportType')
                    if itype == 'SemanticModel':
                        scan_models[it['id']] = dict(tables=it.get('tables') or [], expressions=it.get('expressions') or [])
                    for u in it.get('users') or []:
                        right = next((v for kk, v in u.items() if kk.endswith('UserAccessRight')), None)
                        access.append(_principal(u, 'item', wid, it['id'], it.get('name'), row['item_type'], right))
    if scan_failed:
        notes.append('Scanner failed for ' + '; '.join(scan_failed))
    if not scan_models and any(i['item_type'] == 'SemanticModel' for i in items.values()):
        notes.append('The scanner returned no model tables: turn on the tenant settings "Enhance admin APIs responses with '
                     'detailed metadata" and "... with DAX and mashup expressions".')

by_type = defaultdict(int)
for it in items.values():
    by_type[it['item_type']] += 1
print(f'{len(workspaces)} of {len(all_ws)} workspaces match {LINEAGE["workspaces"]}, excluding {LINEAGE["exclude"]}')
print('Items: ' + ', '.join(f'{t} {n}' for t, n in sorted(by_type.items(), key=lambda x: -x[1])))
if not DRY_RUN:
    print(f'Scanner: {len(scan_models)} models, {len(access)} access rows' + (f', {len(scan_failed)} batch(es) failed' if scan_failed else ''))

In [ ]:
# ── Table catalog: every table in the matching lakehouses (OneLake folders), shortcuts, and the name resolver ────────
#   The lakehouse tables API does not support schema-enabled lakehouses, so tables are listed from the Tables folder:
#   Tables/<schema>/<table> when the lakehouse has schemas, Tables/<table> when it does not.
lakehouses   = [i for i in items.values() if i['item_type'] == 'Lakehouse' and match(i['item'], LINEAGE['table_lakehouses'])]
lh_by_id     = {i['item_id']: i for i in items.values() if i['item_type'] == 'Lakehouse'}
endpoint_lh  = {}      # SQL analytics endpoint id → lakehouse id (Direct Lake models point at the endpoint)
catalog      = []
cat_failed   = []


def _lake_tables(lh):
    ws, lid = lh['workspace_id'], lh['item_id']
    props = fab.call('GET', f'/workspaces/{ws}/lakehouses/{lid}').get('properties') or {}
    sep = props.get('sqlEndpointProperties') or {}
    lh['sql_server'], lh['endpoint_id'], lh['has_schemas'] = sep.get('connectionString'), sep.get('id'), bool(props.get('defaultSchema'))
    base = f'abfss://{ws}@onelake.dfs.fabric.microsoft.com/{lid}/Tables'
    rows, tops = [], [f for f in notebookutils.fs.ls(base) if f.isDir]
    pairs = []
    if lh['has_schemas']:
        for s in tops:
            pairs += [(s.name.rstrip('/'), t.name.rstrip('/')) for t in notebookutils.fs.ls(s.path) if t.isDir]
    else:
        pairs = [('', t.name.rstrip('/')) for t in tops]
    short = {}
    try:
        for sc in fab.pages(f'/workspaces/{ws}/items/{lid}/shortcuts'):
            ol = (sc.get('target') or {}).get('oneLake') or {}
            segs = [x for x in (sc.get('path') or '').strip('/').split('/') if x]
            if segs[:1] == ['Tables'] and ol:
                short[((segs[1] if len(segs) > 1 else ''), sc['name'])] = ol
    except Exception:
        pass                                                   # shortcuts are a bonus: the tables still count
    for schema, table in pairs:
        ol = short.get((schema, table)) or {}
        tsegs = [x for x in (ol.get('path') or '').strip('/').split('/') if x][1:]
        rows.append(dict(workspace_id=ws, workspace=lh['workspace'], lakehouse_id=lid, lakehouse=lh['item'],
                         schema=schema, table=table, table_key=f"{lh['workspace']}/{lh['item']}/{schema}/{table}",
                         is_shortcut=bool(ol), target_workspace_id=ol.get('workspaceId'), target_lakehouse_id=ol.get('itemId'),
                         target_schema=(tsegs[0] if len(tsegs) > 1 else '') if ol else None,
                         target_table=(tsegs[-1] if tsegs else None) if ol else None, target_key=None))
    return rows


if not DRY_RUN:
    with ThreadPoolExecutor(LINEAGE['threads']) as pool:
        futures = {pool.submit(_lake_tables, lh): lh for lh in lakehouses}
    for f, lh in futures.items():
        try:
            catalog += f.result()
        except Exception as e:
            cat_failed.append(dict(workspace=lh['workspace'], item=lh['item'], type='Lakehouse', error=plain(e)))
    for lh in lakehouses:
        if lh.get('endpoint_id'):
            endpoint_lh[lh['endpoint_id']] = lh['item_id']
    by_loc = {(c['lakehouse_id'], c['schema'].lower(), c['table'].lower()): c for c in catalog}
    for c in catalog:                                          # a shortcut points at another catalog table: keep its key
        if c['is_shortcut']:
            t = by_loc.get((c['target_lakehouse_id'], (c['target_schema'] or '').lower(), (c['target_table'] or '').lower()))
            c['target_key'] = t['table_key'] if t else None
    print(f'Catalog: {len(catalog)} tables in {len(lakehouses) - len(cat_failed)} lakehouses'
          + (f', {len(cat_failed)} lakehouse(s) not readable' if cat_failed else '')
          + f', {sum(c["is_shortcut"] for c in catalog)} shortcuts')

cat_by_name = defaultdict(list)
for c in catalog:
    cat_by_name[c['table'].lower()].append(c)
ALIASES = [(re.compile(rx), rep) for rx, rep in LINEAGE.get('name_aliases', [])]


def resolve_ref(r, ctx_ws=None, ctx_lh_ids=()):
    """A parsed reference → [(catalog row, resolution)]. [] means the name is not a catalog table.
    resolution: exact (lakehouse given) | default lakehouse | same workspace | by name | ambiguous."""
    if r.get('dynamic') or not r.get('table'):
        return []
    name = r['table']
    cands = cat_by_name.get(name.lower(), [])
    for rx, rep in ALIASES:                                    # e.g. a _MLV1 build name renamed to _MLV
        if cands:
            break
        cands = cat_by_name.get(rx.sub(rep, name).lower(), [])
    if not cands:
        return []
    lh, ws, schema = r.get('lh_id') or r.get('lh'), r.get('ws'), r.get('schema')
    if lh:
        x = [c for c in cands if lh.lower() in (c['lakehouse_id'].lower(), c['lakehouse'].lower())]
        if x and ws:
            x = [c for c in x if ws.lower() in (c['workspace_id'].lower(), c['workspace'].lower())] or x
        if x:
            if schema:
                x = [c for c in x if c['schema'].lower() == schema.lower()] or x
            return [(c, 'exact' if len(x) == 1 else 'ambiguous') for c in x]
        if any(lh.lower() in (l['item_id'].lower(), (l['item'] or '').lower()) for l in lakehouses):
            return []                                          # a catalog lakehouse was named and the table isn't in it
    if schema:                                                 # 'a.b': schema.table, or lakehouse.table without schemas
        cands = [c for c in cands if schema.lower() in (c['schema'].lower(), c['lakehouse'].lower())] or cands
    x = [c for c in cands if c['lakehouse_id'] in set(ctx_lh_ids or ())]
    if x:
        return [(c, 'default lakehouse') for c in x]
    x = [c for c in cands if ctx_ws and c['workspace_id'] == ctx_ws]
    if len(x) == 1:
        return [(x[0], 'same workspace')]
    cands = x or cands
    return [(cands[0], 'by name')] if len(cands) == 1 else [(c, 'ambiguous') for c in cands]

In [ ]:
# ── Definitions: read every notebook, pipeline, model and report, and parse it ───────────────────────────────────────
#   getDefinition needs Contributor (or higher) on the workspace: Fabric admin alone is not enough.
#   An item that can't be read is listed with the reason; a model's tables then come from the scanner.
FORMAT   = {'Notebook': '?format=ipynb', 'SemanticModel': '?format=TMDL'}
to_read  = sorted((i for i in items.values() if i['item_type'] in LINEAGE['item_types']
                   and not (i['item_type'] == 'Report' and 'paginated' in (i.get('report_type') or '').lower())),
                  key=lambda i: (i['workspace'], i['item_type'], i['item'] or ''))
paginated = [i for i in items.values() if i['item_type'] == 'Report' and 'paginated' in (i.get('report_type') or '').lower()]
refs, links, model_tables, report_fields, not_read, nb_info = [], [], [], [], [], {}
names_rx = build_names_rx({c['table'] for c in catalog}, LINEAGE.get('mention_min_len', 4))
nb_by_name = {(i['workspace_id'], (i['item'] or '').lower()): i for i in items.values() if i['item_type'] == 'Notebook'}
model_by_name = {(i['workspace_id'], (i['item'] or '').lower()): i for i in items.values() if i['item_type'] == 'SemanticModel'}


def _fetch(it):
    try:
        d = fab.call('POST', f"/workspaces/{it['workspace_id']}/items/{it['item_id']}/getDefinition"
                             f"{FORMAT.get(it['item_type'], '')}").get('definition') or {}
        parts = def_parts(d)
        if not parts:
            raise RuntimeError('empty definition returned')
        return parts, None
    except Exception as e:
        return None, plain(e)


def _item_cols(it, prefix='item'):
    return {f'{prefix}_workspace': it['workspace'], f'{prefix}_id': it['item_id'], prefix: it['item'],
            f'{prefix}_type': it['item_type']}


def add_refs(it, found, ctx_ws=None, ctx_lh_ids=()):
    seen = set()
    for r in found:
        k = (r['direction'], r['match_type'], r['evidence'], r['raw'])
        if k in seen:                                          # the same name twice in one cell or activity
            continue
        seen.add(k)
        hits = resolve_ref(r, ctx_ws, ctx_lh_ids)
        base = dict(_item_cols(it), direction=r['direction'], match_type=r['match_type'], evidence=r['evidence'],
                    raw=r['raw'][:500], dynamic=r['dynamic'])
        if not hits:
            refs.append(dict(base, resolution='dynamic' if r['dynamic'] else 'not in catalog'))
        for c, how in hits:
            refs.append(dict(base, resolution=how, table_key=c['table_key'], workspace=c['workspace'],
                             lakehouse=c['lakehouse'], schema=c['schema'], table=c['table']))


def add_link(it, target, link, evidence, target_name=None):
    links.append(dict(_item_cols(it), link=link, evidence=evidence,
                      target_workspace=target['workspace'] if target else None,
                      target_id=target['item_id'] if target else None,
                      target=(target['item'] if target else target_name), target_type=target['item_type'] if target else None))


def model_lakehouse(src):
    """Direct Lake / import source → lakehouse id: endpoint id, OneLake lakehouse id, or the lakehouse name."""
    for v in (src.get('lh'), src.get('database')):
        if not v:
            continue
        if v in endpoint_lh:
            return endpoint_lh[v]
        if v in lh_by_id:
            return v
        by_name = [i for i in lh_by_id.values() if (i['item'] or '').lower() == v.lower()]
        if len(by_name) == 1:
            return by_name[0]['item_id']
    return None


def add_model(it, sources, how_suffix=''):
    for s in sources:
        lid = model_lakehouse(s)
        hits = resolve_ref(dict(table=s.get('table'), schema=s.get('schema'), lh_id=lid, dynamic=False),
                           ctx_ws=it['workspace_id']) if s.get('table') else []
        row = dict(_item_cols(it, 'model'), model_table=s['model_table'], mode=s.get('mode'), columns=s.get('columns'),
                   measures=s.get('measures'), source_table=s.get('table'), source_schema=s.get('schema'))
        if not hits:
            model_tables.append(dict(row, resolution=('not in catalog' if s.get('table') else 'no source') + how_suffix))
        for c, how in hits:
            model_tables.append(dict(row, resolution=how + how_suffix, table_key=c['table_key'], workspace=c['workspace'],
                                     lakehouse=c['lakehouse'], schema=c['schema'], table=c['table']))
            refs.append(dict(_item_cols(it), direction='source', match_type=f"model {s.get('mode') or ''}".strip(),
                             evidence=f"model table {s['model_table']}", raw=s.get('table'), dynamic=False,
                             resolution=how + how_suffix, table_key=c['table_key'], workspace=c['workspace'],
                             lakehouse=c['lakehouse'], schema=c['schema'], table=c['table']))


def scanner_sources(model_id):
    """Fallback when a model's definition can't be read: the scanner's table list and source expressions."""
    sm = scan_models.get(model_id) or {}
    out = []
    for t in sm.get('tables') or []:
        expr = ' '.join((s or {}).get('expression') or '' for s in t.get('source') or [])
        src = m_source(expr)
        out.append(dict(model_table=t.get('name'), mode=t.get('storageMode') or '', columns=len(t.get('columns') or []),
                        measures=len(t.get('measures') or []), **dict(src, table=src['table'] or t.get('name'))))
    return out


if not DRY_RUN:
    with ThreadPoolExecutor(LINEAGE['threads']) as pool:
        fetched = list(zip(to_read, pool.map(_fetch, to_read)))
    for it, (parts, err) in fetched:
        t = it['item_type']
        if err:
            not_read.append(dict(workspace=it['workspace'], item=it['item'], type=t, error=err))
            if t == 'SemanticModel':
                add_model(it, scanner_sources(it['item_id']), ' (scanner)')
            continue
        try:
            if t == 'Notebook':
                text = next((v for k, v in parts.items() if k.endswith('.ipynb')), None) or next(iter(parts.values()))
                found, nlinks, info = parse_notebook(text, LINEAGE.get('helper_patterns', []))
                ctx = [x for x in [info['default_lh_id']] + info['known_lh_ids'] if x]
                code = '\n'.join(''.join(c.get('source') or '') for c in json.loads(text).get('cells') or []
                                 if c.get('cell_type') == 'code')
                named = {r['table'] for r in found if not r['dynamic'] and r.get('table')}
                named |= {rx.sub(rep, n) for rx, rep in ALIASES for n in named}
                explicit = {n.lower() for n in named}
                found += mention_refs(code, names_rx, explicit, 'code')
                add_refs(it, found, it['workspace_id'], ctx)
                nb_info[it['item_id']] = info
                for l in nlinks:
                    add_link(it, nb_by_name.get((it['workspace_id'], l['target'].lower())), l['link'], l['evidence'], l['target'])
            elif t == 'DataPipeline':
                text = next((v for k, v in parts.items() if k.endswith('pipeline-content.json')), None) or next(iter(parts.values()))
                found, plinks = parse_pipeline(text)
                add_refs(it, found, it['workspace_id'])
                for l in plinks:
                    add_link(it, items.get(l['target_id']), l['link'], l['evidence'],
                             '(dynamic)' if l['dynamic'] else l['target_id'])
            elif t == 'SemanticModel':
                tbls, exprs = parse_tmdl(parts)
                add_model(it, model_sources(tbls, exprs))
            elif t == 'Report':
                fields, mref = parse_report(parts)
                model_id = it.get('model_id') or mref.get('model_id')
                if not model_id and mref.get('model_path'):           # byPath: '../SM_AUDIT.SemanticModel'
                    nm = re.sub(r'\.SemanticModel$', '', mref['model_path'].rstrip('/').split('/')[-1], flags=re.I)
                    model_id = (model_by_name.get((it['workspace_id'], nm.lower())) or {}).get('item_id')
                it['model_id'] = model_id
                for f in fields:
                    report_fields.append(dict(_item_cols(it, 'report'), model_id=model_id,
                                              model=(items.get(model_id) or {}).get('item'), **f))
                it['fields_read'] = True
        except Exception as e:
            not_read.append(dict(workspace=it['workspace'], item=it['item'], type=t, error='could not parse: ' + str(e)[:150]))
    for it in items.values():                                  # report → model links (scanner or definition)
        if it['item_type'] == 'Report' and it.get('model_id'):
            add_link(it, items.get(it['model_id']), 'uses model', 'report dataset', it['model_id'])
    print(f'Definitions: {len(to_read) - len(not_read)} of {len(to_read)} read; {len(refs)} table references, '
          f'{len(model_tables)} model tables, {len(report_fields)} report fields')

In [ ]:
# ── Activity log (admin): who viewed which report, one UTC day per call ─────────────────────────────────────────────
#   The API keeps 30 days. Each complete day is loaded once and appended, so the history grows past 30 days.
#   Today is never loaded (still incomplete): today's views appear after tomorrow's run.
#   Limit: about 200 activity-log calls an hour. A day that fails stops the load; the next run carries on from there.
act_new, act_days_new = [], []
if RUN_ACTIVITY and not DRY_RUN:
    loaded = {(str(r['day']), r['activity']) for r in read_rows('lineage_activity_days')}
    days = [TODAY - timedelta(days=k) for k in range(min(LINEAGE['activity_days'], 30), 0, -1)]
    stop = False
    for d in days:
        for act in LINEAGE['activities']:
            if (d.isoformat(), act) in loaded:
                continue
            got, seen = [], set()
            url = (f"/admin/activityevents?startDateTime='{d.isoformat()}T00:00:00.000Z'"
                   f"&endDateTime='{d.isoformat()}T23:59:59.999Z'&$filter=Activity eq '{act.lower()}'")
            try:
                while url:
                    j = pbi('GET', url)
                    for e in j.get('activityEventEntities') or []:
                        if e.get('Id') in seen:
                            continue
                        seen.add(e.get('Id'))
                        got.append(dict(event_id=e.get('Id'), day=d, event_time=e.get('CreationTime'),
                                        activity=e.get('Activity'), user=(e.get('UserId') or '').lower() or None,
                                        workspace_id=e.get('WorkspaceId'), workspace=e.get('WorkSpaceName'),
                                        report_id=e.get('ReportId'), report=e.get('ReportName'),
                                        model_id=e.get('DatasetId'), model=e.get('DatasetName'),
                                        consumption=e.get('ConsumptionMethod'), distribution=e.get('DistributionMethod')))
                    url = None if j.get('lastResultSet') else j.get('continuationUri')
            except Exception as e:
                notes.append(f'Activity log stopped at {d.isoformat()} ({act}): {plain(e)}. The next run carries on from that day.')
                stop = True
                break
            act_new += got
            act_days_new.append(dict(day=d, activity=act, events=len(got)))
        if stop:
            break
    ACT_COLS = ['event_id:s', 'day:d', 'event_time:t', 'activity:s', 'user:s', 'workspace_id:s', 'workspace:s',
                'report_id:s', 'report:s', 'model_id:s', 'model:s', 'consumption:s', 'distribution:s']
    if act_new:
        write_rows('lineage_activity', act_new, ACT_COLS, mode='append')
    if act_days_new:
        write_rows('lineage_activity_days', act_days_new, ['day:d', 'activity:s', 'events:l'], mode='append')
    print(f'Activity log: {len(act_new)} events from {len({r["day"] for r in act_days_new})} new day(s)')

In [ ]:
# ── SQL endpoint query history: who queried which table through the SQL analytics endpoint ─────────────────────────
#   queryinsights.exec_requests_history keeps 30 days. Each lakehouse is read from the day after its last load up to
#   yesterday, and the query text is matched against that lakehouse's table names (whole word, any case).
#   Needs Contributor (or higher) on the lakehouse's workspace. Power BI DirectQuery / Direct Lake fallback queries
#   show up here under the viewer's login too.
sql_new, sql_days_new = [], []
focus_lh = set()
if RUN_SQL and not DRY_RUN:
    try:
        import pyodbc, struct
    except ImportError:
        pyodbc = None
        notes.append('SQL endpoint history skipped: pyodbc is not available in this Spark environment.')
    focus_lh = {c['lakehouse_id'] for c in catalog if is_focus(c)}
    endpoints = [lh for lh in lakehouses if lh.get('sql_server')
                 and (match(lh['item'], LINEAGE['sql_endpoints']) or lh['item_id'] in focus_lh)]
    last = {}
    for r in read_rows('lineage_sql_days'):
        last[r['lakehouse_id']] = max(last.get(r['lakehouse_id'], r['day']), r['day'])
    oldest = TODAY - timedelta(days=30)
    Q = """SELECT login_name, CAST(start_time AS date) AS d, CAST(command AS varchar(8000)) AS cmd,
                  COUNT(*) AS n, MAX(start_time) AS last_time
           FROM queryinsights.exec_requests_history
           WHERE start_time >= ? AND start_time < ?
           GROUP BY login_name, CAST(start_time AS date), CAST(command AS varchar(8000))"""

    def _connect(lh):
        err = None
        for aud in ('pbi', 'https://database.windows.net/'):  # the endpoint accepts either audience; try both
            try:
                tok = notebookutils.credentials.getToken(aud).encode('utf-16-le')
                return pyodbc.connect(f"Driver={{ODBC Driver 18 for SQL Server}};Server={lh['sql_server']},1433;"
                                      f"Database={lh['item']};Encrypt=Yes;TrustServerCertificate=No",
                                      attrs_before={1256: struct.pack('<I', len(tok)) + tok}, timeout=60)
            except Exception as e:
                err = e
        raise err

    for lh in endpoints if pyodbc else []:
        frm = max(oldest, (last[lh['item_id']] + timedelta(days=1)) if lh['item_id'] in last else oldest)
        if frm >= TODAY:
            continue
        tables = {c['table'].lower(): c for c in catalog if c['lakehouse_id'] == lh['item_id']}
        rx = build_names_rx({c['table'] for c in tables.values()}, 1, ignore_case=True)
        try:
            conn = _connect(lh)
            rows = conn.cursor().execute(Q, datetime.combine(frm, datetime.min.time()),
                                         datetime.combine(TODAY, datetime.min.time())).fetchall()
            conn.close()
        except Exception as e:
            notes.append(f"SQL endpoint {lh['workspace']}/{lh['item']}: {plain(e)}")
            continue
        agg = {}
        for login, d, cmd, n, lt in rows:
            low = (cmd or '').lower()
            if rx is None or 'queryinsights' in low:            # this scan's own query
                continue
            for t in {m.group(0).lower() for m in rx.finditer(cmd or '')}:
                k = ((login or '').lower(), d, t)
                a = agg.setdefault(k, [0, lt])
                a[0] += int(n)
                a[1] = max(a[1], lt) if a[1] and lt else (a[1] or lt)
        for (login, d, t), (n, lt) in agg.items():
            c = tables[t]
            sql_new.append(dict(table_key=c['table_key'], workspace=c['workspace'], lakehouse=c['lakehouse'],
                                schema=c['schema'], table=c['table'], user=login, day=d, queries=n, last_time=lt))
        sql_days_new.append(dict(lakehouse_id=lh['item_id'], lakehouse=lh['item'], day=TODAY - timedelta(days=1),
                                 queries=len(rows)))
    SQL_COLS = ['table_key:s', 'workspace:s', 'lakehouse:s', 'schema:s', 'table:s', 'user:s', 'day:d', 'queries:l', 'last_time:t']
    if sql_new:
        write_rows('lineage_sql_usage', sql_new, SQL_COLS, mode='append')
    if sql_days_new:
        write_rows('lineage_sql_days', sql_days_new, ['lakehouse_id:s', 'lakehouse:s', 'day:d', 'queries:l'], mode='append')
    print(f'SQL endpoints: {len(sql_days_new)} of {len(endpoints)} read, {len(sql_new)} table × user × day rows')

In [ ]:
# ── Build lineage_table_usage (lakehouse table × user × path) and write every snapshot table ──────────────────────
#   report view : the user viewed a report (activity log) → its model → the model tables it uses → source table
#   sql query   : the user queried the table through the SQL endpoint
#   access      : the user (or group) can open a report or model that reads the table (item permission or workspace role)
#   owner       : creator / last editor of a notebook or pipeline that reads or writes the table
#   A table reached through a shortcut is also counted against the table the shortcut points at (via_shortcut).
usage = {}
if not DRY_RUN:
    window = TODAY - timedelta(days=LINEAGE['usage_days'])
    cat_by_key = {c['table_key']: c for c in catalog}
    model_src = defaultdict(list)                             # model id → [(model table, catalog row, resolution)]
    for m in model_tables:
        if m.get('table_key'):
            model_src[m['model_id']].append((m['model_table'], cat_by_key[m['table_key']], m['resolution']))
    report_tables = defaultdict(set)                          # report id → model tables its visuals and filters use
    for f in report_fields:
        report_tables[f['report_id']].add(f['table'])

    def tables_for(report_id, model_id):
        """Source tables behind what a report uses; all of the model's tables when the report could not be read."""
        src = model_src.get(model_id) or []
        rep = items.get(report_id) or {}
        if report_id and rep.get('fields_read'):
            used = {t.lower() for t in report_tables.get(report_id, ())}
            return [(c, how, '') for mt, c, how in src if mt.lower() in used]
        return [(c, how, 'all model tables (report fields not read)') for mt, c, how in src]

    def add_use(c, user, path, via, detail='', events=0, last_seen=None, resolution='', model=None, ptype=None):
        if not user:
            return
        for tgt, short in [(c, None)] + ([(cat_by_key[c['target_key']], c['table_key'])] if c.get('target_key') in cat_by_key else []):
            k = (tgt['table_key'], user.lower(), path, (via or {}).get('item_id'), short)
            u = usage.setdefault(k, dict(table_key=tgt['table_key'], workspace=tgt['workspace'], lakehouse=tgt['lakehouse'],
                                         schema=tgt['schema'], table=tgt['table'], user=user.lower(), principal_type=ptype,
                                         path=path, via_type=(via or {}).get('item_type'), via_item=(via or {}).get('item'),
                                         via_workspace=(via or {}).get('workspace'), model=model, detail=detail,
                                         events=0, last_seen=None, resolution=resolution, via_shortcut=short))
            u['events'] += int(events or 0)
            if last_seen and (u['last_seen'] is None or to_ts(last_seen) > to_ts(u['last_seen'])):
                u['last_seen'] = last_seen

    # report view (activity history, last usage_days)
    views = defaultdict(lambda: [0, None])
    for e in read_rows('lineage_activity'):
        if e.get('day') and e['day'] >= window and e.get('user'):
            k = (e['user'], e.get('report_id'), e.get('model_id'), e.get('activity'))
            views[k][0] += 1
            views[k][1] = max(filter(None, [views[k][1], e.get('event_time')]), default=None)
    for (user, rid, mid, act), (n, last) in views.items():
        rep = items.get(rid) or {}
        mid = rep.get('model_id') or mid
        via = rep or items.get(mid)
        for c, how, note in tables_for(rid if rep else None, mid):
            add_use(c, user, 'report view', via, ' · '.join(x for x in [act, note] if x), n, last, how,
                    (items.get(mid) or {}).get('item'), 'User')

    # sql query (last usage_days)
    for s in read_rows('lineage_sql_usage'):
        if s.get('day') and s['day'] >= window and s.get('table_key') in cat_by_key:
            add_use(cat_by_key[s['table_key']], s['user'], 'sql query', None, 'SQL analytics endpoint', s['queries'],
                    s.get('last_time'), 'exact', None, 'User')

    # access: item permissions on reports and models, plus the workspace roles where they sit
    ws_users = defaultdict(list)
    for a in access:
        if a['scope'] == 'workspace':
            ws_users[a['workspace_id']].append(a)
    item_users = defaultdict(list)
    for a in access:
        if a['scope'] == 'item':
            item_users[a['item_id']].append(a)
    for it in items.values():
        if it['item_type'] == 'Report':
            srcs = tables_for(it['item_id'], it.get('model_id'))
        elif it['item_type'] == 'SemanticModel':
            srcs = [(c, how, '') for mt, c, how in model_src.get(it['item_id'], [])]
        else:
            continue
        if not srcs:
            continue
        people = [(a, f"{it['item_type'].lower()}: {a['access_right']}") for a in item_users[it['item_id']]]
        people += [(a, f"workspace role: {a['access_right']}") for a in ws_users[it['workspace_id']]]
        for a, right in people:
            for c, how, note in srcs:
                add_use(c, a['principal'], 'access', it, ' · '.join(x for x in [right, note] if x), 0, None, how,
                        (items.get(it.get('model_id')) or {}).get('item') if it['item_type'] == 'Report' else it['item'],
                        a['principal_type'])

    # owner: who built or last changed a notebook or pipeline that touches the table
    for r in refs:
        if r.get('table_key') and r['item_type'] in ('Notebook', 'DataPipeline'):
            it = items[r['item_id']]
            for who, role in ((it.get('created_by'), 'created'), (it.get('modified_by'), 'last modified')):
                add_use(cat_by_key[r['table_key']], who, 'owner', it, f"{role} · {r['direction']}", 0,
                        it.get('modified_at'), r['resolution'], None, 'User')

    # ── Write the snapshot tables (overwrite every run) ─────────────────────────────────────────────────────────────
    REF_COLS = ['item_workspace:s', 'item_id:s', 'item:s', 'item_type:s', 'direction:s', 'match_type:s', 'evidence:s',
                'raw:s', 'dynamic:b', 'resolution:s', 'table_key:s', 'workspace:s', 'lakehouse:s', 'schema:s', 'table:s']
    written = {
        'lineage_items': write_rows('lineage_items', list(items.values()),
            ['workspace_id:s', 'workspace:s', 'item_id:s', 'item:s', 'item_type:s', 'state:s', 'created_by:s',
             'modified_by:s', 'modified_at:t', 'model_id:s', 'report_type:s']),
        'lineage_tables': write_rows('lineage_tables', catalog,
            ['table_key:s', 'workspace_id:s', 'workspace:s', 'lakehouse_id:s', 'lakehouse:s', 'schema:s', 'table:s',
             'is_shortcut:b', 'target_workspace_id:s', 'target_lakehouse_id:s', 'target_schema:s', 'target_table:s', 'target_key:s']),
        'lineage_refs': write_rows('lineage_refs', refs, REF_COLS),
        'lineage_item_links': write_rows('lineage_item_links', links,
            ['item_workspace:s', 'item_id:s', 'item:s', 'item_type:s', 'link:s', 'evidence:s', 'target_workspace:s',
             'target_id:s', 'target:s', 'target_type:s']),
        'lineage_model_tables': write_rows('lineage_model_tables', model_tables,
            ['model_workspace:s', 'model_id:s', 'model:s', 'model_type:s', 'model_table:s', 'mode:s', 'columns:l', 'measures:l',
             'source_schema:s', 'source_table:s', 'resolution:s', 'table_key:s', 'workspace:s', 'lakehouse:s', 'schema:s', 'table:s']),
        'lineage_report_fields': write_rows('lineage_report_fields', report_fields,
            ['report_workspace:s', 'report_id:s', 'report:s', 'report_type:s', 'model_id:s', 'model:s', 'page:s', 'visual:s',
             'visual_type:s', 'table:s', 'field:s', 'kind:s']),
        'lineage_access': write_rows('lineage_access', access,
            ['workspace_id:s', 'workspace:s', 'item_id:s', 'item:s', 'item_type:s', 'scope:s', 'principal:s',
             'principal_name:s', 'principal_type:s', 'user_type:s', 'access_right:s']),
        'lineage_not_read': write_rows('lineage_not_read', not_read + cat_failed, ['workspace:s', 'item:s', 'type:s', 'error:s']),
        'lineage_table_usage': write_rows('lineage_table_usage', list(usage.values()),
            ['table_key:s', 'workspace:s', 'lakehouse:s', 'schema:s', 'table:s', 'user:s', 'principal_type:s', 'path:s',
             'via_type:s', 'via_item:s', 'via_workspace:s', 'model:s', 'detail:s', 'events:l', 'last_seen:t',
             'resolution:s', 'via_shortcut:s']),
    }
    print('Written: ' + ', '.join(f'{k} {v}' for k, v in written.items()))

In [ ]:
# ── End-of-run report ─────────────────────────────────────────────────────────
line = '═' * 100
print(line)
print(f'Lineage and usage scan {TODAY.isoformat()}{"  [DRY RUN]" if DRY_RUN else ""}')
print(line)
print(f'Workspaces : {len(workspaces)} matched {LINEAGE["workspaces"]}, excluding {LINEAGE["exclude"]}')
print('Items      : ' + ', '.join(f'{t} {n}' for t, n in sorted(by_type.items(), key=lambda x: -x[1])))
if DRY_RUN:
    print('Dry run    : the admin item list works. Nothing else was read or written. Set DRY_RUN = False to scan.')
else:
    print(f'Output     : {out_path("lineage_*")}' + ('' if ROOT else '  (the attached default lakehouse)'))
    print(f'Catalog    : {len(catalog)} tables in {len(lakehouses) - len(cat_failed)} lakehouses matching {LINEAGE["table_lakehouses"]}')
    print(f'Read       : {len(to_read) - len(not_read)} of {len(to_read)} notebooks, pipelines, models and reports'
          + (f' ({len(paginated)} paginated reports listed, not parsed)' if paginated else ''))
    print(f'Found      : {sum(1 for r in refs if r.get("table_key"))} table references, {len(links)} item links, '
          f'{len(model_tables)} model tables, {len(report_fields)} report fields, {len(access)} access rows')
    print(f'Activity   : {len(act_new)} new events; usage counts the last {LINEAGE["usage_days"]} days')
    print(f'SQL        : {len(sql_new)} new table × user × day rows')

    # tables nobody reads: no notebook / pipeline read or mention, no model, no report view, no SQL query
    used = {r['table_key'] for r in refs if r.get('table_key') and r['direction'] in ('read', 'mention', 'source')}
    used |= {u['table_key'] for u in usage.values() if u['path'] in ('report view', 'sql query')}
    used |= {c['target_key'] for c in catalog if c['table_key'] in used and c.get('target_key')}
    unused = defaultdict(list)
    for c in catalog:
        if c['table_key'] not in used:
            unused[f"{c['workspace']}/{c['lakehouse']}"].append(f"{c['schema'] + '.' if c['schema'] else ''}{c['table']}")
    print(f'\nNo reads, no model and no views in {LINEAGE["usage_days"]} days: {sum(map(len, unused.values()))} tables'
          ' (candidates to review, not to drop: a config-driven notebook can read a table by a name built at run time)')
    for lh in sorted(unused):
        names = sorted(unused[lh])
        print(f'  {lh} ({len(names)}): ' + ', '.join(names[:12]) + (f', … +{len(names) - 12}' if len(names) > 12 else ''))

    top = defaultdict(set)
    for u in usage.values():
        if u['path'] in ('report view', 'sql query'):
            top[u['table_key']].add(u['user'])
    if top:
        print(f'\nTop 20 tables by people who used them (report views + SQL queries, last {LINEAGE["usage_days"]} days):')
        for k, users in sorted(top.items(), key=lambda x: -len(x[1]))[:20]:
            print(f'  {len(users):>5}  {k}')

    if FOCUS_TABLES:
        ftabs = sorted((c for c in catalog if is_focus(c)), key=lambda c: c['table_key'])
        print(f'\nWho uses the focus tables {FOCUS_TABLES}: {len(ftabs)} tables'
              f' (report views and SQL queries: last {LINEAGE["usage_days"]} days)')
        if not ftabs:
            print('  No catalog table matches. Check the pattern and that the lakehouse matches table_lakehouses.')
        sql_read = {lh['item_id'] for lh in lakehouses if lh.get('sql_server')
                    and (match(lh['item'], LINEAGE['sql_endpoints']) or lh['item_id'] in focus_lh)} if RUN_SQL else set()
        for c in ftabs:
            rows = [u for u in usage.values() if u['table_key'] == c['table_key']]
            print(f"  {c['table_key']}" + ('' if c['lakehouse_id'] in sql_read else '  (SQL history not read)'))
            if not rows:
                print('      no users found on any path')
            for p in ('report view', 'sql query', 'owner', 'access'):
                ppl = defaultdict(int)
                for u in rows:
                    if u['path'] == p:
                        ppl[u['user']] += u['events'] or 0
                if not ppl:
                    continue
                top_p = sorted(ppl.items(), key=lambda x: (-x[1], x[0]))
                txt = ', '.join(f'{w} ({n})' if p in ('report view', 'sql query') else w for w, n in top_p[:10])
                print(f'      {p:<12}{len(ppl):>4}: {txt}' + (f', … +{len(ppl) - 10}' if len(ppl) > 10 else ''))
        if ftabs:
            print('  Full list: lineage_table_usage, filtered on these table_key values.')

    dyn_only = [items[i] for i, info in nb_info.items()
                if info['dynamic'] and not any(r.get('table_key') and r['item_id'] == i for r in refs)]
    if dyn_only:
        print(f'\nNotebooks whose table names are all built at run time ({len(dyn_only)}): add a helper_patterns rule to see them')
        for it in sorted(dyn_only, key=lambda i: (i['workspace'], i['item']))[:20]:
            print(f'  {it["workspace"]} / {it["item"]}')
    if not_read or cat_failed:
        bad = not_read + cat_failed
        print(f'\nNot read ({len(bad)}):')
        for r in bad[:40]:
            print(f'  {r["workspace"]} / {r["item"]} ({r["type"]}): {r["error"]}')
        if len(bad) > 40:
            print(f'  … and {len(bad) - 40} more (table lineage_not_read)')
for n in notes:
    print(f'\nNote: {n}')
print(line)